# Phase D — independent visual milestone review
Review dense seed crops **without consulting audio**. Existing Phase B crack intervals are shown for context and cannot be edited here. For `major_opening` or `radicle_emergence`, mark the last definitely negative frame and first definitely positive frame, then save. Leave an event absent if the images are ambiguous. The saved interval is `(last_negative, first_positive]`; intermediate frames may remain uncertain. No pixel-change score is used to assign events.


In [ ]:
from pathlib import Path
import sys
PROJECT=Path.cwd().resolve()
if PROJECT.name=='notebooks': PROJECT=PROJECT.parent
if not (PROJECT/'src/phase_d_visual.py').exists(): raise RuntimeError('Open inside seed-germination')
sys.path.insert(0,str(PROJECT))
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from src.phase_d_visual import initialize_milestones, seed_frames, roi_for_seed, save_visual_milestone
from src.dense_visual import load_crop
from io import BytesIO
TABLES=PROJECT/'results/tables/targeted_multimodal'
EVENTS=TABLES/'refined_visual_events.csv'
MILESTONES=TABLES/'visual_milestones.csv'
INVENTORY=TABLES/'dense_window_image_inventory.csv'
ROIS=TABLES/'target_seed_rois.csv'
FRAME_ROIS=TABLES/'target_seed_frame_rois.csv'
milestones=initialize_milestones(EVENTS,MILESTONES)
print('Imported Phase B crack intervals:',sum(milestones.event_type.eq('crack_onset')))


In [ ]:
seed=widgets.Dropdown(options=[f'seed_{i:02d}' for i in range(1,6)],description='Seed:')
event=widgets.Dropdown(options=['major_opening','radicle_emergence'],description='Milestone:')
slider=widgets.IntSlider(description='Frame:',continuous_update=False,layout=widgets.Layout(width='750px'))
negative=widgets.Button(description='Mark last negative')
positive=widgets.Button(description='Mark first positive')
confidence=widgets.Dropdown(options=['','high','medium','low'],description='Confidence:')
notes=widgets.Textarea(description='Notes:',layout=widgets.Layout(width='800px',height='70px'))
save=widgets.Button(description='Save reviewed interval',button_style='success')
viewer=widgets.Output(); status=widgets.Output()
selection={'last':None,'first':None}
frames=None
def load_seed(*_):
    global frames
    frames=seed_frames(INVENTORY,seed.value)
    selection.update(last=None,first=None)
    slider.max=len(frames)-1; slider.value=0
    render()
def render(*_):
    if frames is None:return
    row=frames.iloc[slider.value]
    bounds=roi_for_seed(ROIS,FRAME_ROIS,seed.value,row.timestamp)
    crop=load_crop(row.source_path,bounds,padding=0); crop.thumbnail((900,700))
    buffer=BytesIO();crop.save(buffer,format='PNG')
    crack=pd.read_csv(MILESTONES,keep_default_na=False)
    crack=crack.loc[crack.seed_id.eq(seed.value)&crack.event_type.eq('crack_onset')].iloc[0]
    with viewer:
        clear_output(wait=True)
        print('Timestamp:',row.timestamp,'| frame:',slider.value+1,'/',len(frames))
        print('Saved Phase B crack interval:',crack.last_negative_timestamp,'→',crack.first_positive_timestamp)
        print('Selected',event.value,'last negative:',selection['last'],'| first positive:',selection['first'])
        display(widgets.Image(value=buffer.getvalue(),format='png'))
def mark_negative(_):selection['last']=str(frames.iloc[slider.value].timestamp);render()
def mark_positive(_):selection['first']=str(frames.iloc[slider.value].timestamp);render()
def save_selection(_):
    with status:
        clear_output(wait=True)
        try:
            if not selection['last'] or not selection['first']:raise ValueError('Mark both definite boundary frames first')
            save_visual_milestone(MILESTONES,INVENTORY,seed.value,event.value,selection['last'],selection['first'],confidence.value,notes.value)
            print('Saved with backup. Phase B crack interval unchanged.')
        except Exception as error:print(type(error).__name__+': '+str(error))
seed.observe(load_seed,names='value');event.observe(load_seed,names='value');slider.observe(render,names='value')
negative.on_click(mark_negative);positive.on_click(mark_positive);save.on_click(save_selection)
display(widgets.VBox([widgets.HBox([seed,event]),slider,widgets.HBox([negative,positive]),confidence,notes,save,status,viewer]))
load_seed()
